# Logistic Regression Implementation from Scratch

This notebook implements a logistic regression model using gradient descent to predict Titanic passenger survival based on various features.

## Mathematical Background

### Hypothesis Function (Sigmoid)
The model predicts the probability that the output $y = 1$ using the sigmoid function:
$$ f_{w,b}(x) = g(w \cdot x + b) $$
Where $g(z)$ is the sigmoid function:
$$ g(z) = \frac{1}{1 + e^{-z}} $$
This maps any real number to the range $(0, 1)$, which can be interpreted as a probability.

### Cost Function (Binary Cross-Entropy / Log Loss)
We measure the error using the Binary Cross-Entropy cost:
$$ J(w, b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(f_{w,b}(x^{(i)})) + (1 - y^{(i)}) \log(1 - f_{w,b}(x^{(i)})) \right] $$
Where $m$ is the number of training examples.

### Gradient Descent
To minimize the cost function, we update the parameters iteratively:

**Update Rules:**
$$ w_j := w_j - \alpha \frac{\partial J}{\partial w_j} $$
$$ b := b - \alpha \frac{\partial J}{\partial b} $$

**Gradients:**
$$ \frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} (f_{w,b}(x^{(i)}) - y^{(i)}) x_j^{(i)} $$
$$ \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (f_{w,b}(x^{(i)}) - y^{(i)}) $$

Where $\alpha$ is the learning rate.

## 1. Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import copy

## 2. Data Loading and Exploration

We load the Titanic dataset and perform basic exploration to understand the data structure.

In [ ]:
# Load the dataset
titanic_data = pd.read_csv('./train.csv')

# Display first few rows
titanic_data.head()

In [ ]:
# Basic info about the dataset
titanic_data.info()
titanic_data.describe(include='all')

In [ ]:
# Check for missing values, duplicates, and unique counts
print("Missing values:\n", titanic_data.isnull().sum())
print("\nDuplicates:", titanic_data.duplicated().sum())
print("\nUnique values per column:\n", titanic_data.nunique())

## 3. Data Preprocessing

### 3.1 Handling Missing Values and Dropping Irrelevant Columns
We fill missing `Age` values with the median, `Embarked` with the mode, and drop `Cabin`, `Name`, and `Ticket` columns which are either too sparse or not useful for prediction.

In [ ]:
# Fill missing values
titanic_data['Age'] = titanic_data['Age'].fillna(titanic_data['Age'].median())
titanic_data['Embarked'] = titanic_data['Embarked'].fillna(titanic_data['Embarked'].mode()[0])

# Drop irrelevant columns
titanic_data.drop(columns=['Cabin'], inplace=True)
titanic_data.drop(columns=['Name', 'Ticket'], inplace=True)

### 3.2 Encoding Categorical Variables
Machine learning models require numerical input. We convert categorical columns (`Sex`, `Embarked`) into numeric values.

In [ ]:
# Identify categorical columns
print(f"Categorical columns: {list(titanic_data.select_dtypes(include=['str']).columns)}")

# Sex: male = 0, female = 1
print("Sex unique values:", titanic_data['Sex'].unique())
titanic_data['Sex'] = titanic_data['Sex'].replace(
    titanic_data['Sex'].unique(), 
    list(range(titanic_data['Sex'].nunique()))
)

# Embarked: S = 0, C = 1, Q = 2
print("Embarked unique values:", titanic_data['Embarked'].unique())
titanic_data['Embarked'] = titanic_data['Embarked'].replace(
    titanic_data['Embarked'].unique(), 
    list(range(titanic_data['Embarked'].nunique()))
)

titanic_data.head()

### 3.3 Feature Scaling (Z-Score Normalization)

We normalize features to have a mean of 0 and a standard deviation of 1. This helps gradient descent converge faster.

Formula:
$$ x_{norm} = \frac{x - \mu}{\sigma} $$
Where $\mu$ is the mean and $\sigma$ is the standard deviation.

In [ ]:
def zscore_normalize_features(X):
    """
    Normalizes features using Z-score normalization.
    """
    X = np.array(X, dtype=np.float64)   
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    # Avoid division by zero if sigma is 0
    sigma[sigma == 0] = 1
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

# Prepare training data
X_train = np.array(titanic_data.drop(columns=['Survived']))
y_train = np.array(titanic_data['Survived'])

# Normalize features
X_train, mu, sigma = zscore_normalize_features(X_train)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

## 4. Implementing the Model

### 4.1 Sigmoid Function

The sigmoid function maps any real-valued number to a value between 0 and 1, which we interpret as the probability of the positive class.

Formula:
$$ g(z) = \frac{1}{1 + e^{-z}} $$

In [ ]:
def sigmoid(z):
    """
    Compute the sigmoid of z

    Args:
        z (ndarray): A scalar, numpy array of any size.

    Returns:
        g (ndarray): sigmoid(z), with the same shape as z
    """
    g = 1 / (1 + np.exp(-z))
    return g

### 4.2 Cost Function (Binary Cross-Entropy)

Calculates the cost using the Binary Cross-Entropy (Log Loss) formula.

Formula:
$$ J(w, b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(f_{w,b}(x^{(i)})) + (1 - y^{(i)}) \log(1 - f_{w,b}(x^{(i)})) \right] $$

In [ ]:
def compute_cost_logistic(X, y, w, b):
    """
    Computes cost (Binary Cross-Entropy) for logistic regression.

    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters  
      b (scalar)       : model parameter
      
    Returns:
      cost (scalar): cost
    """
    m = X.shape[0]
    cost = 0.0
    for i in range(m):
        z_i = np.dot(X[i], w) + b
        f_wb_i = sigmoid(z_i)
        cost += -y[i] * np.log(f_wb_i) - (1 - y[i]) * np.log(1 - f_wb_i)
             
    cost = cost / m
    return cost

### 4.3 Compute Gradient

Calculates the partial derivatives of the cost function with respect to $w$ and $b$.

**Formulas:**
$$ \frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} (f_{w,b}(x^{(i)}) - y^{(i)}) x_j^{(i)} $$
$$ \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (f_{w,b}(x^{(i)}) - y^{(i)}) $$

In [ ]:
def compute_gradient_logistic(X, y, w, b): 
    """
    Computes the gradient for logistic regression.
 
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)): target values
      w (ndarray (n,)): model parameters  
      b (scalar)      : model parameter
    Returns
      dj_dw (ndarray (n,)): The gradient of the cost w.r.t. the parameters w. 
      dj_db (scalar)      : The gradient of the cost w.r.t. the parameter b. 
    """
    m, n = X.shape
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):
        f_wb_i = sigmoid(np.dot(X[i], w) + b)
        err_i = f_wb_i - y[i]
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err_i * X[i, j]
        dj_db = dj_db + err_i
    dj_dw = dj_dw / m
    dj_db = dj_db / m
        
    return dj_db, dj_dw

### 4.4 Gradient Descent

Iteratively updates the parameters $w$ and $b$ to minimize the cost function.

**Update Rules:**
$$ w := w - \alpha \frac{\partial J}{\partial w} $$
$$ b := b - \alpha \frac{\partial J}{\partial b} $$

Where $\alpha$ is the learning rate.

In [ ]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters): 
    """
    Performs batch gradient descent for logistic regression.
    
    Args:
      X (ndarray (m,n))   : Data, m examples with n features
      y (ndarray (m,))   : target values
      w_in (ndarray (n,)): Initial values of model parameters  
      b_in (scalar)      : Initial values of model parameter
      alpha (float)      : Learning rate
      num_iters (scalar) : number of iterations to run gradient descent
      
    Returns:
      w (ndarray (n,))   : Updated values of parameters
      b (scalar)         : Updated value of parameter
      J_history (list)   : Cost at each iteration (for plotting)
    """
    J_history = []
    w = copy.deepcopy(w_in)
    b = b_in
    
    for i in range(num_iters):
        # Calculate the gradient
        dj_db, dj_dw = compute_gradient_logistic(X, y, w, b)   

        # Update Parameters
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               
      
        # Save cost J at each iteration
        if i < 100000:
            J_history.append(compute_cost_logistic(X, y, w, b))

        # Print cost every 10% of iterations
        if i % math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]}")
        
    return w, b, J_history

## 5. Training the Model

We initialize weights and bias, then run gradient descent.

In [ ]:
# Initialize fitting parameters
initial_w = np.zeros(X_train.shape[1])
initial_b = 0.

# Hyperparameters
iterations = 2000
alpha = 0.01

# Run gradient descent
w, b, J_history = gradient_descent(X_train, y_train, initial_w, initial_b, alpha, iterations)

print(f"\nw: {w} b: {b}")

## 6. Visualization and Evaluation

### 6.1 Cost Function Convergence
Plotting the cost $J$ over iterations helps verify that the algorithm is converging.

In [ ]:
# Plot cost versus iteration
plt.figure(figsize=(10, 6))
plt.plot(J_history)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Cost vs. Iteration")
plt.grid(True)
plt.show()

### 6.2 Feature Importance
Visualize the weights ($w$) to see which features have the most impact on survival prediction.

In [ ]:
feature_names = titanic_data.drop(columns=['Survived']).columns
plt.figure(figsize=(10, 6))
plt.barh(feature_names, w, color='skyblue')
plt.xlabel("Weight Value (Importance)")
plt.title("Impact of Each Feature on Survival Prediction")
plt.show()

### 6.3 Model Accuracy
Calculate the accuracy of the model on the training set using a threshold of 0.5.

In [ ]:
# Get predictions and convert probabilities to class labels
y_pred_prob = predict(X_train, w, b)
y_pred = (y_pred_prob >= 0.5).astype(int)

# Calculate accuracy
accuracy = np.mean(y_pred == y_train) * 100
print(f"Training Accuracy: {accuracy:.2f}%")

## 7. Making New Predictions

Function to predict the survival probability of a new passenger given their raw features.

In [ ]:
def predict(X, w, b):
    """
    Predict the probability of survival using learned logistic regression parameters w and b.
    
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      w (ndarray (n,)): model parameters  
      b (scalar)      : model parameter
      
    Returns:
      p (ndarray (m,)): Probabilities for each example (values between 0 and 1)
    """
    m, n = X.shape
    p = np.zeros(m)
    
    for i in range(m):
        z_i = np.dot(X[i], w) + b
        f_wb_i = sigmoid(z_i)
        p[i] = f_wb_i

    return p

In [ ]:
def predict_passenger(raw_data, w, b, mu, sigma):
    """
    Takes raw passenger features, normalizes them, and returns a survival prediction.
    
    Args:
      raw_data (list or np.array): The features of the passenger
      w, b: Your trained model parameters
      mu, sigma: Mean and StdDev from your training set
    """
    # Convert to numpy array
    x_input = np.array(raw_data)
    
    # Normalize using training statistics
    x_norm = (x_input - mu) / sigma
    
    # Compute prediction probability
    probability = sigmoid(np.dot(x_norm, w) + b)
    
    # Classify based on threshold of 0.5
    prediction = 1 if probability >= 0.5 else 0
    
    return prediction, probability

In [ ]:
# Example predictions (adjust feature order to match your dataset columns)
# Columns: PassengerId, Pclass, Sex, Age, SibSp, Parch, Fare, Embarked
passenger1 = [1, 3, 0, 22.0, 1, 0, 7.25, 0]  # Male, 3rd class, age 22
passenger2 = [2, 1, 1, 38.0, 1, 0, 71.28, 1] # Female, 1st class, age 38

pred1, prob1 = predict_passenger(passenger1, w, b, mu, sigma)
pred2, prob2 = predict_passenger(passenger2, w, b, mu, sigma)

print(f"--- Prediction Result ---")
print(f"Passenger 1: {'Survived' if pred1 == 1 else 'Did not survive'} (Probability: {prob1:.4f})")
print(f"Passenger 2: {'Survived' if pred2 == 1 else 'Did not survive'} (Probability: {prob2:.4f})")